<a href="https://colab.research.google.com/github/maithili39/AI/blob/main/Smart_Email_Reply_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Smart Email Reply Agent

This agent will read emails, generate context-aware replies, and aim to learn your tone over time. We'll start by setting up the Groq API and creating a basic version of the agent.

In [10]:
# Install the Groq Python SDK
!pip install groq

### Set up Groq API Key

To use the Groq API, you'll need an API key. If you don't already have one, create a key from the [Groq Console](https://console.groq.com/keys).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Name the secret `GROQ_API_KEY`. Then, we'll access it securely in the code below.

In [11]:
import os
from groq import Groq
from google.colab import userdata

# Set your Groq API key from Colab secrets
GROQ_API_KEY = #
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initialize the Groq client
client = Groq()

### Define the Email Agent's Persona and System Prompt

The system prompt is crucial for guiding the agent's behavior. It defines its role, tone, and objectives when generating email replies.

### Pushing to GitHub from Colab using a Personal Access Token (PAT)

This section demonstrates how to authenticate with GitHub using a Personal Access Token (PAT) stored in Colab secrets to push changes from your Colab environment to a GitHub repository.

### Important Considerations:

*   **Replace Placeholders**: Make sure to replace `"your_email@example.com"`, `"Your GitHub Username"`, `"your_username"`, and `"your_repo"` with your actual GitHub details.
*   **Existing Repositories**: If you're working with an already cloned repository, you'll need to navigate into its directory (`%cd /content/your_repo_name`) and potentially update its remote URL to include the PAT as shown.
*   **Security**: While using Colab secrets is secure, be mindful of the scopes you grant to your PAT. Grant only the necessary permissions.
*   **Branch Name**: Ensure you push to the correct branch (e.g., `main` or `master`).

In [12]:
system_prompt = """You are a professional email assistant. Your goal is to generate concise, polite, and helpful email replies based on the context of the incoming email. Adopt a tone that is professional yet friendly, and always strive to provide clear, actionable information. If a direct answer isn't possible, politely state so and suggest next steps.

Keep your replies to a maximum of 3-4 sentences unless more detail is explicitly requested or necessary. Start with a polite greeting and end with a professional closing. Avoid jargon unless the context explicitly requires it.
"""

### Simulate an Incoming Email

Since we can't directly read your emails, we'll simulate an incoming email for the agent to reply to. You can modify this `incoming_email_content` with any email you want the agent to respond to.

In [13]:
incoming_email_content = """Subject: Project Update Request

Hi Team,

Could you please provide an update on the status of the 'Apollo' project by end of day today? I need to report back to stakeholders.

Thanks,
Sarah
"""

### Generate the Email Reply

Now, we'll use the Groq API with our defined system prompt and the simulated email to generate a reply.

In [16]:
try:
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": f"Please draft a reply to the following email:\n\n{incoming_email_content}",
            }
        ],
        model="llama-3.1-8b-instant", # You can choose other Groq models like 'llama3-8b-8192' or 'llama3-70b-8192'
        temperature=0.7, # Adjust temperature for creativity vs. consistency
        max_tokens=200, # Limit reply length
    )

    generated_reply = chat_completion.choices[0].message.content
    print("Generated Email Reply:")
    print(generated_reply)

except Exception as e:
    print(f"An error occurred: {e}")

Generated Email Reply:
Subject: Re: Project Update Request

Dear Sarah,

I'd be happy to provide an update on the Apollo project. I will compile the necessary information and send it to you by the end of the day today. If you have any specific requirements or questions, please let me know and I'll be happy to assist.

Best regards,
[Your Name]


### Create an Interactive GUI with `ipywidgets`

Now, let's create a simple interactive interface using `ipywidgets` so you can easily test different incoming emails and system prompts.

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Input widget for incoming email content
email_input = widgets.Textarea(
    value=incoming_email_content, # Pre-fill with the last used email
    placeholder='Paste your incoming email here',
    description='Incoming Email:',
    disabled=False,
    continuous_update=False,
    layout=widgets.Layout(width='80%', height='150px')
)

# Input widget for system prompt (editable)
system_prompt_input = widgets.Textarea(
    value=system_prompt, # Pre-fill with the last used system prompt
    placeholder='Define the agent\'s persona here',
    description='System Prompt:',
    disabled=False,
    continuous_update=False,
    layout=widgets.Layout(width='80%', height='100px')
)

# Button to generate reply
generate_button = widgets.Button(
    description='Generate Reply',
    disabled=False,
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to generate an email reply',
    icon='envelope'
)

# Output widget to display the generated reply
output_area = widgets.Output()

# Function to call when the button is clicked
def on_button_click(b):
    with output_area:
        clear_output() # Clear previous output
        print("Generating reply...")
        try:
            current_system_prompt = system_prompt_input.value
            current_incoming_email = email_input.value

            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": current_system_prompt,
                    },
                    {
                        "role": "user",
                        "content": f"Please draft a reply to the following email:\n\n{current_incoming_email}",
                    }
                ],
                model="llama-3.1-8b-instant",
                temperature=0.7,
                max_tokens=200,
            )

            generated_reply = chat_completion.choices[0].message.content
            print("\nGenerated Email Reply:")
            print(generated_reply)

        except Exception as e:
            print(f"An error occurred: {e}")

# Attach the click event to the button
generate_button.on_click(on_button_click)

# Display the widgets
display(system_prompt_input, email_input, generate_button, output_area)


Textarea(value="You are a professional email assistant. Your goal is to generate concise, polite, and helpful …

Textarea(value="Subject: Project Update Request\n\nHi Team,\n\nCould you please provide an update on the statu…

Button(button_style='success', description='Generate Reply', icon='envelope', style=ButtonStyle(), tooltip='Cl…

Output()